# 02 -- dim_contract Seed

**Purpose**: Build `dim_contract`, the contract-type reference table.
`contract_id` is the unique primary key (1st, 2nd, 3rd ... FA). ETL joins
`fact_team.contract_id` to this table to read the exact `cap_hit_pct` and
`guaranteed` flag for a player's current status -- no arithmetic needed.

**Key**: `contract_id` (unique)

**Outputs**:
- `data/dim_contract.parquet` (10 rows)

In [1]:
import pandas as pd
import numpy as np
import nflreadpy as nfl
from pathlib import Path
from datetime import datetime, date
from dataclasses import dataclass, field


@dataclass
class LeagueConfig:
    draft_year: int = 2026
    total_cap: int = 500_000_000
    num_teams: int = 28
    num_conferences: int = 2
    initial_contract_years: int = 3        # years 1-3 (Fixed Salary)
    extension_contract_years: int = 3      # years 4-6 (New Value)
    fa_minimum_salary: int = 2_000_000     # FA row: League Minimum Salary
    # Cap hit % lives in dim_contract rows; never hardcode here.
    data_dir: str = "data"
    fuzzy_auto_threshold: int = 90
    fuzzy_review_threshold: int = 70


CFG = LeagueConfig()
DATA = Path(CFG.data_dir)
TODAY = date.today().isoformat()
DATA.mkdir(exist_ok=True)

## 1 -- Contract Type Definitions

Each entry in `_CONTRACT_DEFS` expands into `total_years` rows.
Add a new tuple here if a new contract type is introduced — never hardcode
cap percentages in roster ETL logic.

In [2]:
# -- Contract definitions: one tuple per final row ───────────────────────────
# contract_id  : unique PK; the label shown in UI (1st / 2nd / ... / FA)
# contract_type: grouping field (initial / extension / franchise / expiring / minor / fa)
# contract_label: label grouping field (Initial / Extension / Franchise Tag / Expiring (X) / Minor / Free Agent)
# contract_year : 1/2/3 position within the term; used by ETL to advance contracts
# total_years  : full length of this contract type
# cap_hit_pct  : fraction of contract_value charged to cap for this row
# guaranteed   : True = dead money applies if player is dropped
# cap_exempt   : True = does not count against the salary cap
# min_salary   : dollar floor; None for all non-FA types
# notes        : display note matching the league rule sheet
_CONTRACT_DEFS = [
    # contract_id,     contract_type, salary_type,              contract_year, total_years,
    #   cap_hit_pct, guaranteed, cap_exempt, min_salary, notes

    # -- Initial term (years 1-3, Fixed Salary) --------------------------------
    ("1st", "initial",   "Initial",   "Fixed Salary",          1, CFG.initial_contract_years,
     0.50, True,  False, None, ""),
    ("2nd", "initial",   "Initial",   "Fixed Salary",          2, CFG.initial_contract_years,
     0.40, True,  False, None, "Re-valued post-deadline"),
    ("3rd", "initial",   "Initial",   "Fixed Salary",          3, CFG.initial_contract_years,
     0.00, False, False, None, ""),

    # -- Extension term (years 4-6, New Value) ---------------------------------
    ("4th", "extension",   "Extension",   "New Value",              1, CFG.extension_contract_years,
     0.50, True,  False, None, "Extension period"),
    ("5th", "extension",   "Extension",   "New Value",              2, CFG.extension_contract_years,
     0.40, True,  False, None, "Extension period"),
    ("6th", "extension",   "Extension",   "New Value",              3, CFG.extension_contract_years,
     0.00, False, False, None, "Final year before UFA"),

    # -- Single-year types -----------------------------------------------------
    ("Tag", "franchise",   "Franchise Tag",   "New Salary",   1, 1,
     0.50, True,  False, None,                  "Expiring contract"),
    ("X",             "expiring",   "Expiring",  "Fixed Salary", 1, 1,
     0.50, False, False, None,                  "Expiring contract"),
    ("Minor",         "minor",   "Minor",     "Fixed Salary", 1, 1,
     0.00, False, True,  None,                  "Exempt from Cap"),
    ("FA",            "fa",   "Free Agent",        "League Minimum Salary", 1, 1,
     0.00, False, True,  CFG.fa_minimum_salary, "Set each year at the end of preseason"),
]


def build_dim_contract(defs: list) -> pd.DataFrame:
    # One tuple = one row; no year explosion needed.
    rows = []
    for (contract_id, contract_type, contract_label, salary_type, contract_year, total_years,
         cap_hit_pct, guaranteed, cap_exempt, min_salary, notes) in defs:
        rows.append({
            "contract_id":   contract_id,
            "contract_type": contract_type,
            "contract_label": contract_label,
            "salary_type":   salary_type,
            "contract_year": contract_year,
            "total_years":   total_years,
            "cap_hit_pct":   cap_hit_pct,
            "guaranteed":    guaranteed,
            "cap_exempt":    cap_exempt,
            "min_salary":    min_salary,
            "notes":         notes,
        })
    return pd.DataFrame(rows)


dim_contract = build_dim_contract(_CONTRACT_DEFS)
out_path = DATA / "dim_contract.parquet"
dim_contract.to_parquet(out_path, index=False)
print(f"dim_contract: {len(dim_contract)} rows -> {out_path}")
dim_contract

dim_contract: 10 rows -> data\dim_contract.parquet


,contract_id,contract_type,contract_label,salary_type,contract_year,total_years,cap_hit_pct,guaranteed,cap_exempt,min_salary,notes
0,1st,initial,Initial,Fixed Salary,1,3,0.5,True,False,NaN,
1,2nd,initial,Initial,Fixed Salary,2,3,0.4,True,False,NaN,Re-valued post-deadline
2,3rd,initial,Initial,Fixed Salary,3,3,0.0,False,False,NaN,
3,4th,extension,Extension,New Value,1,3,0.5,True,False,NaN,Extension period
4,5th,extension,Extension,New Value,2,3,0.4,True,False,NaN,Extension period
5,6th,extension,Extension,New Value,3,3,0.0,False,False,NaN,Final year before UFA
6,Tag,franchise,Franchise Tag,New Salary,1,1,0.5,True,False,NaN,Expiring contract
7,X,expiring,Expiring,Fixed Salary,1,1,0.5,False,False,NaN,Expiring contract
8,Minor,minor,Minor,Fixed Salary,1,1,0.0,False,True,NaN,Exempt from Cap
9,FA,fa,Free Agent,League Minimum Salary,1,1,0.0,False,True,2000000.0,Set each year at the end of preseason


## 2 -- Validation

In [3]:
df = pd.read_parquet(DATA / "dim_contract.parquet")
print(f"dim_contract: {len(df)} rows, {len(df.columns)} columns")
print()
print(df[[
    "contract_id", "contract_type", "contract_label", "salary_type", "contract_year",
    "cap_hit_pct", "guaranteed", "cap_exempt", "notes"
]].to_string(index=False))
print()
# contract_id must be unique
dupes = df["contract_id"].duplicated().sum()
print(f"Duplicate contract_id values: {dupes}  (expected 0)")

dim_contract: 10 rows, 11 columns

contract_id contract_type contract_label           salary_type  contract_year  cap_hit_pct  guaranteed  cap_exempt                                 notes
        1st       initial        Initial          Fixed Salary              1          0.5        True       False                                      
        2nd       initial        Initial          Fixed Salary              2          0.4        True       False               Re-valued post-deadline
        3rd       initial        Initial          Fixed Salary              3          0.0       False       False                                      
        4th     extension      Extension             New Value              1          0.5        True       False                      Extension period
        5th     extension      Extension             New Value              2          0.4        True       False                      Extension period
        6th     extension      Extension       